# PDF File Ingestion with TeradataGenAI Vector Store - Complete Lifecycle Management

This notebook demonstrates **end-to-end PDF file ingestion with collection lifecycle management** using the modern **Ingestor fluent API** for comprehensive document processing.

## 🎯 **What This Notebook Accomplishes**
1. **PDF Document Processing** - Extract text, images, and metadata from PDF files using multiple ingestors
2. **Multi-Cloud Integration** - Ingest from Local Files, S3, Azure Blob, and Google Cloud Storage  
3. **Collection Lifecycle Management** - Create, update, search, and properly destroy collections
4. **Advanced Document Processing** - Compare BasicIngestor, NVIngestor, and UnstructuredIngestor capabilities
5. **Complete Pipeline Demonstrations** - From file ingestion to searchable vector collections

## 📋 **Ingestor API vs REST API Benefits**
- **Before**: Manual REST calls with complex JSON payloads, difficult collection management
- **Now**: Simple fluent API with `.files()`, `.extract()`, `.embed()`, `.create()`, `.run()` and proper cleanup

## 🔄 **Collection Management Strategy**
- **Cleanup First**: Destroy existing collections before creating new ones to avoid conflicts
- **Resource Management**: Properly clean up collections when no longer needed
- **Naming Convention**: Clear, descriptive collection names that indicate purpose and ingestor type

## 🔐 **Security Features**
- **Secure Credential Collection**: Uses getpass for secure credential input
- **No Hardcoded Credentials**: All sensitive information collected at runtime
- **Environment Variables**: Credentials stored securely in environment for session use

---

**🚀 Complete PDF document processing pipeline with proper lifecycle management and secure credentials!**

In [ ]:
# 1. Environment Setup and Auto-Reload Configuration
print("🔧 Setting up file-based vector store environment...")


import os
import tempfile
from pathlib import Path

print("✅ Auto-reload configured - modules will refresh on code changes")
print("📂 Python paths added for development")

# Now import the modules that we want to auto-reload
print("📚 Importing TeradataGenAI modules...")

# TeradataML imports
from teradataml import set_auth_token

# TeradataGenAI file-based imports
import teradatagenai
from teradatagenai import (
    Collection, CollectionManager,
    LocalConfig, S3Config, AzureBlobConfig, GCPConfig,
    BasicIngestor, NVIngestor, UnstructuredIngestor,
    ExtractionSchema, ColumnInfo, HNSW, FLAT, SearchParams,
    Ingestor, CollectionType, TeradataAI
)
from teradatasqlalchemy.types import VARCHAR, INTEGER

print("✅ All imports successful!")
print("🔄 Modules will now auto-reload when you modify source code")
print("📚 Environment ready for file-based vector store creation")

# Get the base directory of the teradatagenai package
base_dir = os.path.dirname(teradatagenai.__file__)

<span style="color: #FFB000;">

## 2. Secure Credential Collection

Collect credentials securely using getpass (no hardcoded values)

</span>

In [ ]:
# 2. Secure Credential Collection
print("🔐 Collecting secure credentials for PDF ingestion...")

from getpass import getpass
import os

# Collect database credentials
print("📊 Please provide database connection details:")

# Using secure credential collection pattern
base_url = getpass("Teradata Base URL: ")
os.environ['TD_USERNAME'] = getpass("Teradata Username: ")
os.environ['TD_PASSWORD'] = getpass("Teradata Password: ")
os.environ['TD_BASE_URL'] = base_url
os.environ['TD_AUTH_MECH'] = 'BASIC'

# Collect ML model/API credentials
print("\n🤖 Please provide ML model service details:")
embeddings_base_url = getpass("Embeddings Model Base URL: ")
nvidia_api_key = getpass("NVIDIA API Key: ")
os.environ["NVIDIA_API_KEY"] = nvidia_api_key

# Collect NV Ingest service details
print("\n🔧 Please provide NV Ingest service details:")
nv_ingest_host = getpass("NV Ingest Host: ")

# Collect cloud storage credentials (optional)
print("\n☁️ Cloud storage credentials (optional - press Enter to skip):")
aws_access_key_id = getpass("AWS Access Key ID (optional): ")
aws_secret_access_key = getpass("AWS Secret Access Key (optional): ")
s3_bucket_name = getpass("S3 Bucket Name (optional): ")

azure_account_name = getpass("Azure Storage Account Name (optional): ")
azure_account_key = getpass("Azure Storage Account Key (optional): ")
azure_container_name = getpass("Azure Container Name (optional): ")

gcp_bucket_name = getpass("GCP Bucket Name (optional): ")

print("\n✅ All credentials collected and stored securely")
print("🔒 Credentials are now available for use throughout the notebook")
print("⚠️ Credentials won't be visible in notebook outputs")
print("🔐 Ready for secure PDF file ingestion")

<span style="color: #FFB000;">

## 3.  Establish authentication token

</span>

In [ ]:
print("🔗 Configuring database connection...")

# Get/Set the authentication token using securely collected credentials
auth_data = set_auth_token(
    base_url=base_url, 
    username=os.environ['TD_USERNAME'], 
    password=os.environ['TD_PASSWORD'], 
    auth_mech=os.environ['TD_AUTH_MECH']
)

print("✅ Database connection established")
print(f"🌐 Connected to: {base_url}")
print(f"👤 User: {os.environ['TD_USERNAME']}")

# Verify service health
try:
    health = CollectionManager.health(return_type="json")
    print("✅ Collection service is healthy")
except Exception as e:
    print(f"ℹ️ Service status: {e}")

<span style="color: #FFB000;">

## 4. Configure PDF File Ingestion Settings

Setup local PDF files and configure multiple ingestors with securely collected credentials

</span>

In [ ]:
# 4. Configure Local File Ingestion - The Modern Way
print("📁 Setting up Local PDF File Ingestion with Ingestor API...")

# Define collection names with clear, descriptive identifiers
local_pdf_collection_name = "pdf_local_basic_ingestor_secure"
local_pdf_nv_collection_name = "pdf_local_nv_ingestor_secure"
local_pdf_unstructured_collection_name = "pdf_local_unstructured_ingestor_secure"

# Configure embedding model using securely collected credentials
pdf_embedding_model = TeradataAI(
    api_type="nim",
    model_name="nvidia/llama-3.2-nv-embedqa-1b-v2",
    api_base=embeddings_base_url
)

local_files = [
    os.path.join(base_dir, 'example-data', 'pdf', 'InDb_Analytical_Functions.pdf'), 
    os.path.join(base_dir, 'example-data', 'pdf', 'multimodal_test.pdf'),
    os.path.join(base_dir, 'example-data', 'pdf', 'SQL_Fundamentals.pdf')
]

# Create LocalConfig for PDF files from VectorStore folder
local_pdf_config = LocalConfig(
    files=local_files,  # Process all .pdf files in VectorStore directory
    files_type="pdf",
    delimiter=None,  # Not needed for PDF files
)

# Configure basic ingestor with chunking parameters for PDF processing
pdf_basic_ingestor = BasicIngestor(
    chunk_size=512,  # Larger chunks for PDF content
    chunk_overlap=150  # More overlap for better context
)

# Advanced NVIngestor with comprehensive extraction using secure credentials
pdf_nv_ingestor = NVIngestor(
    ingest_host=nv_ingest_host,
    ingest_port=443,
    chunk_size=1024,
    chunk_overlap=100,
    extract_text=True,
    extract_images=True
)

pdf_unstructured_ingestor = UnstructuredIngestor(
    chunk_size=512,  # Larger chunks for PDF content
    chunk_overlap=100,
    extract_images=False  # More overlap for better context
)

# Configure extraction schema for storing extracted content
pdf_basic_extraction_schema = ExtractionSchema(
    table_name="pdf_basic_uploads_secure",  # Table name for storing extracted content
    data_columns=[
        ColumnInfo(
            name="text",
            datatype=VARCHAR(32000),
            description="Extracted text content from PDF files using BasicIngestor"
        )
    ]
)

# Custom extraction schema for structured content
pdf_nv_extraction_schema = ExtractionSchema(
    table_name="pdf_nv_ingest_uploads_secure",
    data_columns=[
        ColumnInfo(
            name="text",
            datatype=VARCHAR(32000),
            description="Extracted document content using NVIngestor"
        )
    ],
    metadata_columns=[
        ColumnInfo(
            name="page_number",
            datatype=INTEGER(),
            description="Page number in the PDF"
        ),
        ColumnInfo(
            name="language",
            datatype=VARCHAR(10),
            description="Detected language"
        ),
        ColumnInfo(
            name="text_metadata",
            datatype=VARCHAR(1000),
            description="Text metadata from NVIngestor"
        )
    ]
)

print("✅ Local PDF file configuration complete!")
print(f"📂 Files pattern: {local_pdf_config.files}")
print(f"🔧 Basic ingestor chunk size: {pdf_basic_ingestor.chunk_size} characters")
print(f"🤖 Embedding model: {pdf_embedding_model.model_name}")
print(f"📋 Basic extraction schema: {pdf_basic_extraction_schema.table_name} table with {len(pdf_basic_extraction_schema.data_columns)} data column(s)")
print("🔄 Ready to create file-based collections with multiple PDF Ingestor APIs")
print("🔐 All configurations use securely collected credentials")

---

## 🗂️ Part 1: Local File Ingestion with Basic Processing

**Use Case:** Process local files with automatic text extraction and embedding generation using secure credentials  
**Key Feature:** Simple file ingestion with BasicIngestor for straightforward text processing

### 📊 Before vs After Comparison:
- **REST API Approach**: Manual file handling, complex JSON payloads, multiple API calls
- **Ingestor Approach**: Single fluent pipeline with `.files().embed().create().run()`
- **Security**: No hardcoded credentials, secure runtime collection

In [ ]:
# Initialize collection reference and clean up if exists
print(f"🧹 Cleaning up existing collection: {local_pdf_collection_name}")
local_pdf_basic_collection = Collection(name=local_pdf_collection_name, auth_data=auth_data)

In [ ]:
# Check if collection exists before destroying
local_pdf_basic_collection.status(return_type="json")

In [ ]:
# Destroy the collection if it exists to ensure clean state
print("🗑️ Destroying existing collection to ensure clean state...")
local_pdf_basic_collection.destroy()

In [ ]:
# 5. Create Local PDF Collection with BasicIngestor - Single Pipeline Execution
print("🚀 Creating Local PDF File-Based Collection using BasicIngestor Pipeline...")

try:
    # The modern way - single fluent pipeline that replaces multiple REST calls!
    local_basic_ingestor_result = (
        Ingestor(
            name=local_pdf_collection_name,
            type=CollectionType.FILE_CONTENT_BASED,
            target_database=os.environ['TD_USERNAME'],  # Use secure credentials
            description="Demo: Local PDF file ingestion with BasicIngestor using secure credentials",
            auth_data=auth_data
        )
        # Configure file source and processing
        .files(
            files=local_pdf_config,
            ingestor=pdf_basic_ingestor,
            extraction_schema=pdf_basic_extraction_schema
        )
        # Configure indexing algorithm
        .create(
            embedding_model=pdf_embedding_model 
        )
        # Execute the complete pipeline
        .run()
    )
    
    print("✅ Local PDF file-based collection created successfully!")
    print(f"🎯 Pipeline success: {local_basic_ingestor_result['status']['success']}")
    print(f"🔐 Security: Used secure credentials (no hardcoded values)")
    
    if local_basic_ingestor_result['status']['success']:
        local_pdf_basic_collection = local_basic_ingestor_result["collection"]
        print(f"📚 Collection name: {local_pdf_basic_collection.name}")
        print("🎉 PDF files processed, embeddings generated, and index created!")
    else:
        print(f"⚠️ Pipeline issues: {local_basic_ingestor_result['status']['errors']}")
        
except Exception as e:
    print(f"ℹ️ Collection creation: {e}")
    print("💡 This demonstrates the Ingestor API approach")
    # Create collection reference for testing
    local_pdf_basic_collection = Collection(name=local_pdf_collection_name)

print("\n📊 What just happened with BasicIngestor:")
print("  1️⃣ Created collection with FILE_CONTENT_BASED type")
print("  2️⃣ Ingested PDF files from local directory with BasicIngestor")
print("  3️⃣ Generated embeddings using NVIDIA Llama model")
print("  4️⃣ Built HNSW index for fast similarity search")
print("  🎯 All in a single fluent pipeline - no manual REST calls!")
print("  🔐 All using securely collected credentials")

In [ ]:
# View the BasicIngestor pipeline execution result
local_basic_ingestor_result

In [ ]:
# 6. Test Local PDF Collection with Similarity Search (BasicIngestor)
print("🔍 Testing Local PDF Collection with BasicIngestor Similarity Search...")
pdf_basic_collection = Collection(name=local_pdf_collection_name, auth_data=auth_data)

# Test queries related to actual PDF content (LLM handbook, SQL fundamentals, etc.)
pdf_test_queries = [
    "What is SQL and how do I use it?",
    "Tell me about LLM models and techniques",
    "How do multimodal systems work?",
    "What are the basic PDF functionality features?"
]

for i, query in enumerate(pdf_test_queries, 1):
    print(f"\n{i}️⃣ Query: '{query}'")
    try:
        search_results = pdf_basic_collection.similarity_search(
            question=query,
            search_params=SearchParams(top_k=3), 
            return_type="json"
        )
        print(f"   🔍 Query processed successfully")
        print("   📋 Top results:")
        print(search_results)
        
    except Exception as e:
        print(f"   🔍 Search test: {e}")

print("\n🏁 Local PDF file ingestion validation complete!")
print("✅ Demonstrated: LocalConfig + BasicIngestor + HNSW indexing for PDF files")
print("🔐 All operations completed using secure credentials")
print("🔄 Collection ready for additional ingestion or can be destroyed when no longer needed")

#### Adding Additional PDF Files to Existing Collection

**Security Note:** All file additions use the same secure credentials collected at the beginning

In [ ]:
# Add additional PDF file to existing collection
print("📄 Adding additional PDF file to existing BasicIngestor collection...")

# Use sample PDF file from example-data
additional_pdf_file = os.path.join(base_dir, 'example-data', 'pdf', 'basic-text.pdf')
print(f"📁 Additional file: {os.path.basename(additional_pdf_file)}")

# Create LocalConfig for the additional PDF file
additional_pdf_config = LocalConfig(
    files=additional_pdf_file,
    files_type="pdf"
)

# Configure extraction schema for additional files
additional_pdf_extraction_schema = ExtractionSchema(
    table_name="pdf_additional_uploads_secure",  # Different table for additional files
    data_columns=[
        ColumnInfo(
            name="text",
            datatype=VARCHAR(32000),
            description="Additional PDF text content extracted with BasicIngestor"
        )
    ]
)

print("📝 Adding new PDF file to existing collection...")
# Add files to existing collection using secure credentials
additional_files_result = (
    Ingestor(
        name=local_pdf_collection_name, 
        auth_data=auth_data  # Use secure authentication
    )
    # Configure file source and processing
    .files(
        files=additional_pdf_config,
        ingestor=pdf_basic_ingestor,
        extraction_schema=additional_pdf_extraction_schema
    )
    # Execute the pipeline (no .create() needed as collection exists)
    .run()
)

print("✅ Additional PDF file ingestion complete!")
print("🔐 File addition completed using secure credentials")
print("🔄 Collection now contains files from multiple sources")

In [ ]:
# View additional files ingestion result
additional_files_result

## Update Collection Embeddings After File Ingestion

**Security Note:** Embedding updates use securely collected credentials and models

In [ ]:
# Update collection embeddings after adding new files
print("🔄 Updating collection embeddings after file ingestion...")

collection_update_result = (
    Ingestor(
        name=local_pdf_collection_name, 
        auth_data=auth_data
    )
    .create(
        embedding_model=pdf_embedding_model
    )
    .run()
)

print("✅ Collection embeddings updated successfully!")
print("🎯 All ingested files now have fresh embeddings and are searchable")
print("🔐 Update completed using secure credentials")

In [ ]:
# View collection update result
collection_update_result

In [ ]:
# Test updated collection with new query
print("🔍 Testing updated collection with multi-file content...")
updated_search_result = pdf_basic_collection.similarity_search(
    question="What are basic text formatting options?",
    search_params=SearchParams(top_k=3), 
    return_type="json"
)
print("Search results from updated collection:")
updated_search_result

In [ ]:
# Clean up BasicIngestor collection - no longer needed
print(f"🗑️ Destroying BasicIngestor collection: {local_pdf_collection_name}")
pdf_basic_collection.destroy()
print("✅ BasicIngestor collection destroyed - resources cleaned up")
print("🔐 Cleanup completed using secure credentials")

## Creating PDF Collection with NVIngestor for Advanced Processing

**Security Note:** NVIngestor uses securely collected host credentials

In [ ]:
# 7. Create PDF Collection with NVIngestor - Advanced Pipeline Execution
print("🚀 Creating PDF Collection using NVIngestor for advanced processing...")

# Clean up existing NVIngestor collection first
print(f"🧹 Cleaning up existing collection: {local_pdf_nv_collection_name}")
try:
    Collection(name=local_pdf_nv_collection_name, auth_data=auth_data).destroy()
    print("✅ Existing NVIngestor collection destroyed")
except Exception as e:
    print(f"ℹ️ No existing collection to destroy: {e}")

try:
    # Advanced NVIngestor pipeline with secure credentials
    local_nv_ingestor_result = (
        Ingestor(
            name=local_pdf_nv_collection_name,
            type=CollectionType.FILE_CONTENT_BASED,
            target_database=os.environ['TD_USERNAME'],  # Use secure credentials
            description="Demo: Local PDF file ingestion with NVIngestor using secure credentials",
            auth_data=auth_data
        )
        # Configure file source and advanced processing
        .files(
            files=local_pdf_config,
            ingestor=pdf_nv_ingestor,  # Uses secure NV Ingest host
            extraction_schema=pdf_nv_extraction_schema
        )
        # Configure indexing algorithm
        .create(
            embedding_model=pdf_embedding_model 
        )
        # Execute the complete pipeline
        .run()
    )
    
    print("✅ Local PDF NVIngestor collection created successfully!")
    print(f"🎯 Pipeline success: {local_nv_ingestor_result['status']['success']}")
    print(f"🔐 Security: Used secure NV Ingest host and credentials")
    
except Exception as e:
    print(f"ℹ️ NVIngestor collection creation: {e}")
    print("💡 NVIngestor demonstrates advanced multimodal processing")

---

## Collection Management and Cleanup

**Lifecycle Management:** Following PDF ingestor best practices for proper resource cleanup with secure credentials

**Security Features:**
- All operations use securely collected credentials
- No hardcoded sensitive information
- Credentials managed via environment variables
- Secure NV Ingest host configuration

**Resource Management:**
- Clean up collections when no longer needed
- Monitor collection status
- Handle errors gracefully

**Best Practices:**
- Always destroy existing collections before creating new ones
- Use descriptive collection names indicating security features
- Test with similarity search after ingestion
- Properly handle exceptions and provide meaningful error messages
- **Never hardcode credentials in notebooks**

In [ ]:
# 8. Optional: Cleanup All Collections (Uncomment to destroy collections)
print("🧹 Collection Cleanup Management...")

collections_to_cleanup = [
    local_pdf_collection_name,
    local_pdf_nv_collection_name,
    local_pdf_unstructured_collection_name
]

for collection_name in collections_to_cleanup:
    try:
        collection = Collection(name=collection_name, auth_data=auth_data)
        collection.destroy()
        print(f"✅ Collection '{collection_name}' destroyed successfully")
    except Exception as cleanup_error:
        print(f"ℹ️ Collection '{collection_name}' cleanup: {cleanup_error}")

print(f"🔐 All cleanup operations completed using secure credentials")
print("🎉 PDF ingestion demonstration complete with secure credential management!")